# Simulasi Simbolik VQE 2-Qubit (Depth D=4)

Notebook ini berisi perhitungan aljabar linear untuk sirkuit kuantum menggunakan `sympy` sesuai dengan bedah matematis sirkuit.

In [8]:
from sympy.physics.quantum import TensorProduct
import sympy
from sympy import symbols, Matrix, I, cos, sin, exp, kronecker_product, simplify, init_printing

# Mengaktifkan tampilan LaTeX yang cantik di Jupyter
init_printing(use_latex='mathjax')

def get_ry(theta):
    """Matriks rotasi Ry tunggal"""
    return Matrix([
        [cos(theta/2), -sin(theta/2)],
        [sin(theta/2),  cos(theta/2)]
    ])

def get_rz(theta):
    """Matriks rotasi Rz tunggal"""
    return Matrix([
        [exp(-I*theta/2), 0],
        [0, exp(I*theta/2)]
    ])

# ---------------------------------------------------------
# 1. Definisi Simbol dan Parameter (D=4)
# ---------------------------------------------------------
depth = 4
thetas = {}
for l in range(depth + 1):
    for q in range(2):
        for g in ['y', 'z']:
            name = f'theta_{l}_{q}_{g}'
            thetas[(l, q, g)] = symbols(name, real=True)

print(f"Total parameter: {len(thetas)}")


Total parameter: 20


In [3]:
# ---------------------------------------------------------
# 2. Matriks Lapis Rotasi (U_rot)
# ---------------------------------------------------------
def get_u_rot_layer(l):
    """Menghasilkan matriks 4x4 untuk lapisan rotasi ke-l"""
    # Qubit 0: Rz * Ry
    u0 = get_rz(thetas[(l, 0, 'z')]) * get_ry(thetas[(l, 0, 'y')])
    # Qubit 1: Rz * Ry
    u1 = get_rz(thetas[(l, 1, 'z')]) * get_ry(thetas[(l, 1, 'y')])
    # Tensor product (Q0 \otimes Q1)
    return kronecker_product(u0, u1)

# ---------------------------------------------------------
# 3. Matriks Lapis Keterikatan (U_ent)
# ---------------------------------------------------------
cnot01 = Matrix([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
])

cnot10 = Matrix([
    [1, 0, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0],
    [0, 1, 0, 0]
])

U_ent = cnot10 * cnot01

print("Matriks Entanglement (U_ent):")
display(U_ent)


Matriks Entanglement (U_ent):


⎡1  0  0  0⎤
⎢          ⎥
⎢0  0  1  0⎥
⎢          ⎥
⎢0  0  0  1⎥
⎢          ⎥
⎣0  1  0  0⎦

In [10]:

# ---------------------------------------------------------
# 4. Konstruksi Vektor Keadaan Penuh (Full Statevector)
# ---------------------------------------------------------
psi_0 = Matrix([1, 0, 0, 0])

# Iterasi sirkuit: U_rot(4) * U_ent * U_rot(3) * ... * U_rot(0) * |00>
psi_theta = psi_0
for l in range(depth + 1):
    psi_theta = get_u_rot_layer(l) * psi_theta
    if l < depth:
        psi_theta = U_ent * psi_theta

print(f"\nVektor Keadaan Simbolik (Lapisan 0 saja sebagai contoh):")
display(simplify(get_u_rot_layer(0) * psi_0))

# ---------------------------------------------------------
# 5. Matriks Pengukuran (Observasi Z di Qubit 0)
# ---------------------------------------------------------
sigma_z = Matrix([[1, 0], [0, -1]])
I_2 = Matrix([[1, 0], [0, 1]])
M = kronecker_product(sigma_z, I_2)

print("\nMatriks Pengukuran (sigma_z r'$\otimes$' I")
display(M)

# ---------------------------------------------------------
# 6. Skenario Bell State
# ---------------------------------------------------------
print("\n--- PEMBUKTIAN SKENARIO BELL STATE ---")



<>:23: SyntaxWarning: invalid escape sequence '\o'
<>:23: SyntaxWarning: invalid escape sequence '\o'
/tmp/ipykernel_12095/1323306578.py:23: SyntaxWarning: invalid escape sequence '\o'
  print("\nMatriks Pengukuran (sigma_z r'$\otimes$' I")



Vektor Keadaan Simbolik (Lapisan 0 saja sebagai contoh):


⎡ -ⅈ⋅(θ_0_0_z + θ_0_1_z)                           ⎤
⎢ ───────────────────────                          ⎥
⎢            2               ⎛θ_0_0_y⎞    ⎛θ_0_1_y⎞⎥
⎢ℯ                       ⋅cos⎜───────⎟⋅cos⎜───────⎟⎥
⎢                            ⎝   2   ⎠    ⎝   2   ⎠⎥
⎢                                                  ⎥
⎢ ⅈ⋅(-θ_0_0_z + θ_0_1_z)                           ⎥
⎢ ──────────────────────                           ⎥
⎢           2               ⎛θ_0_1_y⎞    ⎛θ_0_0_y⎞ ⎥
⎢ℯ                      ⋅sin⎜───────⎟⋅cos⎜───────⎟ ⎥
⎢                           ⎝   2   ⎠    ⎝   2   ⎠ ⎥
⎢                                                  ⎥
⎢  ⅈ⋅(θ_0_0_z - θ_0_1_z)                           ⎥
⎢  ─────────────────────                           ⎥
⎢            2              ⎛θ_0_0_y⎞    ⎛θ_0_1_y⎞ ⎥
⎢ ℯ                     ⋅sin⎜───────⎟⋅cos⎜───────⎟ ⎥
⎢                           ⎝   2   ⎠    ⎝   2   ⎠ ⎥
⎢                                                  ⎥
⎢  ⅈ⋅(θ_0_0_z + θ_0_1_z)                      


Matriks Pengukuran (sigma_z r'$\otimes$' I


⎡1  0  0   0 ⎤
⎢            ⎥
⎢0  1  0   0 ⎥
⎢            ⎥
⎢0  0  -1  0 ⎥
⎢            ⎥
⎣0  0  0   -1⎦


--- PEMBUKTIAN SKENARIO BELL STATE ---


In [5]:
print("Skenario 1: Terciptanya Bell State")
v_in_1 = Matrix([1/sympy.sqrt(2), 1/sympy.sqrt(2), 0, 0])
v_out_1 = U_ent * v_in_1
display(v_out_1)

Skenario 1: Terciptanya Bell State


⎡√2⎤
⎢──⎥
⎢2 ⎥
⎢  ⎥
⎢0 ⎥
⎢  ⎥
⎢0 ⎥
⎢  ⎥
⎢√2⎥
⎢──⎥
⎣2 ⎦

In [6]:
print("\nSkenario 2: Penghancuran Keterikatan (Status Separable)")
v_in_2 = Matrix([1/sympy.sqrt(2), 0, 1/sympy.sqrt(2), 0])
v_out_2 = U_ent * v_in_2
display(v_out_2)


Skenario 2: Penghancuran Keterikatan (Status Separable)


⎡√2⎤
⎢──⎥
⎢2 ⎥
⎢  ⎥
⎢√2⎥
⎢──⎥
⎢2 ⎥
⎢  ⎥
⎢0 ⎥
⎢  ⎥
⎣0 ⎦